In [4]:
import numpy as np
import rasterio
import geopandas as gpd
from rasterio.mask import mask
from shapely.geometry import mapping, box
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
from rasterio.features import shapes
from rasterio.warp import transform_bounds
import pandas as pd
import os
from tqdm import tqdm
import warnings
from sklearn.preprocessing import StandardScaler
import joblib

In [5]:
# Define class labels
CLASS_MAPPING = {
    'lake': 1,
    'river': 2,
    'land': 0
}

def load_polygons(shapefile_path, class_field='land_type'):
    """
    Load polygons from a shapefile with class labels
    
    Parameters:
    shapefile_path: Path to the shapefile containing polygons
    class_field: Field name in the shapefile that contains the class labels
    
    Returns:
    GeoDataFrame containing the polygons with class information
    """
    gdf = gpd.read_file(shapefile_path)
    
    # Print column names to help debugging
    print(f"Shapefile columns: {gdf.columns.tolist()}")
    
    # Ensure the GeoDataFrame has the specified class column
    if class_field not in gdf.columns:
        raise ValueError(f"Shapefile must contain a '{class_field}' column with values: lake, river, or land")
    
    # Print unique values in the class field
    print(f"Unique values in {class_field}: {gdf[class_field].unique()}")
    
    # Convert class names to numeric labels
    gdf['label'] = gdf[class_field].map(CLASS_MAPPING)
    
    # Check for missing labels and warn
    missing_labels = gdf[gdf['label'].isna()]
    if not missing_labels.empty:
        warnings.warn(f"Found {len(missing_labels)} polygons with unknown land type values. "
                     f"Expected values: {list(CLASS_MAPPING.keys())}")
        gdf = gdf.dropna(subset=['label'])
    
    return gdf

def get_image_bounds(image_path):
    """
    Get the geographic bounds of a raster image
    
    Parameters:
    image_path: Path to the raster image
    
    Returns:
    Tuple of (bounds, crs)
    """
    with rasterio.open(image_path) as src:
        bounds = src.bounds
        crs = src.crs
    return bounds, crs

def find_overlapping_polygons(gdf, image_bounds, image_crs):
    """
    Find polygons that overlap with the given image bounds
    
    Parameters:
    gdf: GeoDataFrame with polygons
    image_bounds: Bounds of the image (minx, miny, maxx, maxy)
    image_crs: CRS of the image
    
    Returns:
    GeoDataFrame containing only the polygons that overlap with the image
    """
    # Create a box polygon from the image bounds
    minx, miny, maxx, maxy = image_bounds
    image_box = box(minx, miny, maxx, maxy)
    
    # Convert the box to a GeoDataFrame with the image CRS
    image_gdf = gpd.GeoDataFrame(geometry=[image_box], crs=image_crs)
    
    # Reproject the input polygons to match the image CRS if needed
    if gdf.crs != image_crs:
        gdf_reprojected = gdf.to_crs(image_crs)
    else:
        gdf_reprojected = gdf.copy()
    
    # Find polygons that intersect with the image box
    intersecting_mask = gdf_reprojected.geometry.intersects(image_box)
    overlapping_polygons = gdf_reprojected[intersecting_mask].copy()
    
    # For very large polygons, clip them to the image extent to avoid memory issues
    overlapping_polygons['geometry'] = overlapping_polygons.geometry.intersection(image_box)
    
    return overlapping_polygons

In [6]:
def extract_features_from_polygon(bands, polygon, buffer=0):
    """
    Extract features from bands within a specific polygon
    
    Parameters:
    bands: Dictionary of band arrays (blue, green, red, nir, swir)
    polygon: Shapely polygon geometry
    buffer: Optional buffer around polygon (in image units)
    
    Returns:
    X: Feature array for pixels within the polygon
    """
    try:
        # Apply buffer if specified
        if buffer > 0:
            polygon = polygon.buffer(buffer)
        
        # Check if polygon has valid geometry
        if polygon.is_empty:
            return None, None
        
        # Check if we have all required bands
        required_bands = ['blue', 'green', 'red', 'nir']
        if not all(band in bands for band in required_bands):
            missing = [band for band in required_bands if band not in bands]
            print(f"Missing required bands: {missing}")
            return None, None
            
        # Verify transforms are available for all bands
        for band_name, band_data in bands.items():
            if 'transform' not in band_data:
                print(f"Transform missing for {band_name} band")
                return None, None
                
        # Use blue band as reference
        blue_band = bands['blue']
        
        # Mask the bands with the polygon
        try:
            # Create mask for polygon
            masked_data = {}
            for band_name, band_data in bands.items():
                # Set up the source info for masking
                src_info = {
                    'count': 1,
                    'transform': band_data['transform'],
                    'height': band_data['data'].shape[0],
                    'width': band_data['data'].shape[1],
                    'dtype': band_data['data'].dtype
                }
                
                out_image, out_transform = mask(
                    src_info,
                    [mapping(polygon)],
                    crop=True,
                    all_touched=True,
                    nodata=0,
                    filled=True
                )
                
                # Store masked data
                masked_data[band_name] = out_image[0]
            
        except ValueError as e:
            if "Input shapes do not overlap raster" in str(e):
                return None, None
            else:
                print(f"Masking error: {e}")
                return None, None
        
        # Skip if any masked region is empty
        if any(np.all(data == 0) for data in masked_data.values()):
            return None, None
        
        # Extract basic spectral bands
        blue = masked_data['blue']
        green = masked_data['green']
        red = masked_data['red']
        nir = masked_data['nir']
        
        # Calculate NDWI (green - NIR) / (green + NIR)
        ndwi = np.zeros_like(green, dtype=np.float32)
        valid_denom = (green + nir) != 0
        ndwi[valid_denom] = (green[valid_denom] - nir[valid_denom]) / (green[valid_denom] + nir[valid_denom])
        
        # Calculate MNDWI if SWIR band is available
        has_mndwi = 'swir' in masked_data
        if has_mndwi:
            swir = masked_data['swir']
            mndwi = np.zeros_like(green, dtype=np.float32)
            valid_denom = (green + swir) != 0
            mndwi[valid_denom] = (green[valid_denom] - swir[valid_denom]) / (green[valid_denom] + swir[valid_denom])
        else:
            mndwi = np.zeros_like(ndwi)
        
        # Calculate texture (simple standard deviation in a window)
        from scipy.ndimage import generic_filter
        texture = generic_filter(red, np.std, size=3)
        
        # Calculate NDVI (NIR - Red) / (NIR + Red)
        ndvi = np.zeros_like(red, dtype=np.float32)
        valid_denom = (nir + red) != 0
        ndvi[valid_denom] = (nir[valid_denom] - red[valid_denom]) / (nir[valid_denom] + red[valid_denom])
        
        # Create mask for valid pixels (not nodata or nan)
        valid_mask = ~np.isnan(ndwi) & (blue != 0) & (green != 0) & (red != 0) & (nir != 0)
        
        if np.sum(valid_mask) < 10:  # Require at least 10 valid pixels
            return None, None
        
        # Extract features for valid pixels
        features = [
            blue[valid_mask].ravel(),     # Blue
            green[valid_mask].ravel(),    # Green
            red[valid_mask].ravel(),      # Red
            nir[valid_mask].ravel(),      # NIR
            ndwi[valid_mask].ravel()      # NDWI
        ]
        
        # Add MNDWI if available
        if has_mndwi:
            features.append(mndwi[valid_mask].ravel())
        
        # Add texture feature
        features.append(texture[valid_mask].ravel())
        
        # Add NDVI
        features.append(ndvi[valid_mask].ravel())
        
        # Stack features into 2D array (samples × features)
        X = np.vstack(features).T
        
        # Check for NaN values and replace them
        if np.isnan(X).any():
            X = np.nan_to_num(X, nan=0.0)
        
        return X, valid_mask
            
    except Exception as e:
        print(f"Error processing polygon: {e}")
        return None, None
    
def load_bands(band_paths):
    """
    Load all band data from specified paths
    
    Parameters:
    band_paths: Dictionary with keys 'blue', 'green', 'red', 'nir', and optionally 'swir'
                and values as file paths
    
    Returns:
    Dictionary of band data with transform information
    """
    bands = {}
    
    # Load each band
    for band_name, band_path in band_paths.items():
        if band_path and os.path.exists(band_path):
            try:
                with rasterio.open(band_path) as src:
                    # Store band data and metadata separately
                    bands[band_name] = {
                        'data': src.read(1),
                        'transform': src.transform,
                        'crs': src.crs,
                        'profile': src.profile,
                        'width': src.width,
                        'height': src.height
                    }
                print(f"Successfully loaded {band_name} band from {os.path.basename(band_path)}")
            except Exception as e:
                print(f"Error loading {band_name} band from {band_path}: {e}")
        else:
            print(f"Warning: {band_name} band file not found at {band_path}")
    
    # Verify we have all required bands with transform info
    required_bands = ['blue', 'green', 'red', 'nir']
    for band in required_bands:
        if band not in bands:
            print(f"Missing required band: {band}")
        elif 'transform' not in bands[band]:
            print(f"Missing transform information for band: {band}")
    
    return bands

def create_training_dataset(band_paths_list, polygons_gdf):
    """
    Create a training dataset by combining features from multiple images
    
    Parameters:
    band_paths_list: List of dictionaries, each with paths to blue, green, red, nir, and optionally swir bands
    polygons_gdf: GeoDataFrame with polygons and their class labels
    
    Returns:
    X: Feature array
    y: Labels array
    feature_names: Names of the features
    """
    X_list = []
    y_list = []
    has_mndwi = False
    
    # Process each set of bands
    for i, band_paths in enumerate(band_paths_list):
        print(f"Processing image set {i+1}/{len(band_paths_list)}")
        
        # Check if we have the required bands
        required_bands = ['blue', 'green', 'red', 'nir']
        if not all(band in band_paths for band in required_bands):
            print(f"Warning: Image set {i+1} is missing required bands. Skipping.")
            continue
        
        # Check if SWIR is available
        if 'swir' in band_paths and band_paths['swir']:
            has_mndwi = True
            print("SWIR band found, will calculate MNDWI.")
        
        # Load the bands
        bands = load_bands(band_paths)
        
        # Get image bounds and CRS (using blue band as reference)
        if 'blue' in bands:
            blue_band = bands['blue']
            minx, maxx, miny, maxy = rasterio.transform.array_bounds(
                blue_band['data'].shape[0], 
                blue_band['data'].shape[1], 
                blue_band['transform']
            )
            image_bounds = (minx, miny, maxx, maxy)
            image_crs = blue_band['crs']
        else:
            print(f"Warning: Blue band not available for image set {i+1}. Skipping.")
            continue
        
        # Find polygons that overlap with this image
        overlapping_polygons = find_overlapping_polygons(polygons_gdf, image_bounds, image_crs)
        
        if len(overlapping_polygons) == 0:
            print(f"No polygons overlap with image set {i+1}")
            continue
        
        print(f"Found {len(overlapping_polygons)} overlapping polygons")
        
        # Extract features for each overlapping polygon
        for idx, row in tqdm(overlapping_polygons.iterrows(), 
                           total=len(overlapping_polygons), 
                           desc=f"Extracting from image set {i+1}"):
            
            X, valid_mask = extract_features_from_polygon(bands, row.geometry)
            
            if X is not None and X.shape[0] > 0:
                X_list.append(X)
                y_list.append(np.full(X.shape[0], row['label']))
    
    # Check if we have any features
    if len(X_list) == 0:
        raise ValueError("No valid features could be extracted from any of the images")
    
    # Combine all samples
    X = np.vstack(X_list)
    y = np.hstack(y_list)
    
    # Determine feature names based on whether MNDWI was calculated
    if has_mndwi:
        feature_names = ['Blue', 'Green', 'Red', 'NIR', 'NDWI', 'MNDWI', 'Texture', 'NDVI']
    else:
        feature_names = ['Blue', 'Green', 'Red', 'NIR', 'NDWI', 'Texture', 'NDVI']
    
    # Print dataset summary
    print(f"\nTraining dataset created:")
    print(f"  Total samples: {X.shape[0]}")
    print(f"  Features: {X.shape[1]}")
    print(f"  Class distribution:")
    for class_name, class_id in CLASS_MAPPING.items():
        count = np.sum(y == class_id)
        percentage = count / len(y) * 100
        print(f"    {class_name}: {count} samples ({percentage:.1f}%)")
    
    return X, y, feature_names


In [7]:
def train_water_model(X, y, feature_names, model_type='rf'):
    """
    Train a model to classify water bodies with multiple model options
    
    Parameters:
    X: Feature array
    y: Labels array
    feature_names: Names of the features
    model_type: 'rf' for Random Forest, 'svm' for Support Vector Machine, or 'both' for both
    
    Returns:
    Trained model(s), feature importances (for RF), and test metrics
    """
    # Apply feature scaling (especially important for SVM)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # Create dictionary to store models and metrics
    results = {}
    target_names = ["land", "lake", "river"]
    
    # Train and evaluate Random Forest if requested
    if model_type in ['rf', 'both']:
        # Initialize model
        rf_model = RandomForestClassifier(
            n_estimators=100,
            max_depth=15,
            min_samples_split=10,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        )
        
        # Train model
        print("\nTraining Random Forest model...")
        rf_model.fit(X_train, y_train)
        
        # Evaluate
        y_pred_rf = rf_model.predict(X_test)
        accuracy_rf = accuracy_score(y_test, y_pred_rf)
        conf_matrix_rf = confusion_matrix(y_test, y_pred_rf)
        report_rf = classification_report(y_test, y_pred_rf, target_names=target_names)
        
        print(f"Random Forest Accuracy: {accuracy_rf:.3f}")
        print("\nConfusion Matrix:")
        print(conf_matrix_rf)
        print("\nClassification Report:")
        print(report_rf)
        
        # Feature importance
        importances = rf_model.feature_importances_
        indices = np.argsort(importances)[::-1]
        
        print("\nFeature Ranking (Random Forest):")
        for i, idx in enumerate(indices):
            if idx < len(feature_names):
                print(f"{i+1}. {feature_names[idx]} - {importances[idx]:.4f}")
        
        # Store results
        results['rf'] = {
            'model': rf_model,
            'accuracy': accuracy_rf,
            'confusion_matrix': conf_matrix_rf,
            'report': report_rf,
            'feature_importances': dict(zip(feature_names, importances))
        }
    
    # Train and evaluate SVM if requested
    if model_type in ['svm', 'both']:
        # Initialize model
        svm_model = SVC(
            kernel='rbf', 
            C=10, 
            gamma='scale', 
            class_weight='balanced', 
            random_state=42,
            probability=True
        )
        
        # Train model
        print("\nTraining SVM model...")
        svm_model.fit(X_train, y_train)
        
        # Evaluate
        y_pred_svm = svm_model.predict(X_test)
        accuracy_svm = accuracy_score(y_test, y_pred_svm)
        conf_matrix_svm = confusion_matrix(y_test, y_pred_svm)
        report_svm = classification_report(y_test, y_pred_svm, target_names=target_names)
        
        print(f"SVM Accuracy: {accuracy_svm:.3f}")
        print("\nConfusion Matrix:")
        print(conf_matrix_svm)
        print("\nClassification Report:")
        print(report_svm)
        
        # Store results
        results['svm'] = {
            'model': svm_model,
            'accuracy': accuracy_svm,
            'confusion_matrix': conf_matrix_svm,
            'report': report_svm
        }
    
    # Save the scaler with the models
    results['scaler'] = scaler
    results['feature_names'] = feature_names
    
    # Return the appropriate model(s)
    if model_type == 'both':
        return results
    else:
        return results[model_type]

In [8]:
def create_visualization(classification, output_path):
    """
    Create a colored visualization of the classification
    
    Parameters:
    classification: Classification array
    output_path: Path to save the visualization
    """
    # Define colors for each class (land, lake, river)
    colors = {
        0: [0.7, 0.7, 0.7],    # Land - gray
        1: [0.0, 0.5, 0.8],    # Lake - blue
        2: [0.0, 0.3, 0.6]     # River - dark blue
    }
    
    # Create RGB image
    rgb = np.zeros((classification.shape[0], classification.shape[1], 3), dtype=np.float32)
    
    for class_val, color in colors.items():
        mask = classification == class_val
        for i in range(3):
            rgb[:, :, i][mask] = color[i]
    
    # Save the visualization
    plt.figure(figsize=(10, 10))
    plt.imshow(rgb)
    plt.axis('off')
    plt.savefig(output_path, bbox_inches='tight', dpi=300)
    plt.close()


In [9]:
def apply_model_to_image(model_results, band_paths, output_path, model_type='rf', probabilities=False):
    """
    Apply the trained model to a new set of bands
    
    Parameters:
    model_results: Model results dictionary from train_water_model
    band_paths: Dictionary with paths to blue, green, red, nir, and optionally swir bands
    output_path: Where to save the classification result
    model_type: 'rf' or 'svm', which model to use if both were trained
    probabilities: If True, save probability maps instead of class predictions
    
    Returns:
    classification: The classification array
    """
    # Extract the model and scaler
    if model_type == 'both':
        model_type = 'rf'  # Default to RF if both were trained
        print("Using Random Forest model for prediction")
    
    model = model_results[model_type]['model']
    scaler = model_results['scaler']
    feature_names = model_results['feature_names']
    
    # Determine if we should expect MNDWI in our features
    has_mndwi = 'MNDWI' in feature_names
    
    # Check if we have the required bands
    required_bands = ['blue', 'green', 'red', 'nir']
    if not all(band in band_paths for band in required_bands):
        raise ValueError(f"Missing required bands. Need {', '.join(required_bands)}")
    
    # Check MNDWI compatibility
    if has_mndwi and ('swir' not in band_paths or not band_paths['swir']):
        print("Warning: Model was trained with MNDWI but no SWIR band provided. Results may be less accurate.")
    
    try:
        # Load all bands
        bands = load_bands(band_paths)
        
        # Get reference band (blue) for dimensions and transform
        blue_band = bands['blue']
        height, width = blue_band['data'].shape
        transform = blue_band['transform']
        profile = blue_band['profile']
        
        # Prepare output arrays
        classification = np.zeros((height, width), dtype=np.uint8)
        classification.fill(255)  # Fill with nodata value
        
        # Prepare probability arrays if requested
        if probabilities:
            prob_land = np.zeros((height, width), dtype=np.float32)
            prob_lake = np.zeros((height, width), dtype=np.float32)
            prob_river = np.zeros((height, width), dtype=np.float32)
        
        # Process the image in chunks to avoid memory issues
        chunk_size = 1024  # Process in 1024x1024 pixel chunks
        
        # Process the image in chunks
        for y in tqdm(range(0, height, chunk_size), desc="Processing image rows"):
            # Adjust chunk height for edge of image
            chunk_height = min(chunk_size, height - y)
            
            for x in range(0, width, chunk_size):
                # Adjust chunk width for edge of image
                chunk_width = min(chunk_size, width - x)
                
                # Extract chunk for each band
                blue = bands['blue']['data'][y:y+chunk_height, x:x+chunk_width]
                green = bands['green']['data'][y:y+chunk_height, x:x+chunk_width]
                red = bands['red']['data'][y:y+chunk_height, x:x+chunk_width]
                nir = bands['nir']['data'][y:y+chunk_height, x:x+chunk_width]
                
                # Skip if chunk is empty
                if np.all(blue == 0) or np.all(green == 0) or np.all(red == 0) or np.all(nir == 0):
                    continue
                
                # Calculate NDWI
                ndwi = np.zeros_like(green, dtype=np.float32)
                valid_denom = (green + nir) != 0
                ndwi[valid_denom] = (green[valid_denom] - nir[valid_denom]) / (green[valid_denom] + nir[valid_denom])
                
                # Calculate MNDWI if needed and available
                if has_mndwi and 'swir' in bands:
                    swir = bands['swir']['data'][y:y+chunk_height, x:x+chunk_width]
                    mndwi = np.zeros_like(green, dtype=np.float32)
                    valid_denom = (green + swir) != 0
                    mndwi[valid_denom] = (green[valid_denom] - swir[valid_denom]) / (green[valid_denom] + swir[valid_denom])
                else:
                    mndwi = np.zeros_like(green, dtype=np.float32)
                
                # Calculate texture
                from scipy.ndimage import generic_filter
                texture = generic_filter(red, np.std, size=3)
                
                # Calculate NDVI
                ndvi = np.zeros_like(red, dtype=np.float32)
                valid_denom = (nir + red) != 0
                ndvi[valid_denom] = (nir[valid_denom] - red[valid_denom]) / (nir[valid_denom] + red[valid_denom])
                
                # Create mask for valid pixels
                valid_mask = ~np.isnan(ndwi) & (blue != 0) & (green != 0) & (red != 0) & (nir != 0)
                
                if np.sum(valid_mask) == 0:
                    continue
                
                # Prepare feature vector
                features = [
                    blue[valid_mask].ravel(),
                    green[valid_mask].ravel(),
                    red[valid_mask].ravel(),
                    nir[valid_mask].ravel(),
                    ndwi[valid_mask].ravel()
                ]
                
                # Add MNDWI if needed and available
                if has_mndwi:
                    features.append(mndwi[valid_mask].ravel())
                
                # Add texture and NDVI
                features.append(texture[valid_mask].ravel())
                features.append(ndvi[valid_mask].ravel())
                
                # Stack and scale features
                X_chunk = np.vstack(features).T
                X_chunk = np.nan_to_num(X_chunk)  # Replace any remaining NaNs
                X_chunk_scaled = scaler.transform(X_chunk)
                
                # Make predictions
                if probabilities:
                    proba = model.predict_proba(X_chunk_scaled)
                    
                    # Populate probability arrays
                    chunk_proba_land = np.zeros(valid_mask.shape, dtype=np.float32)
                    chunk_proba_lake = np.zeros(valid_mask.shape, dtype=np.float32)
                    chunk_proba_river = np.zeros(valid_mask.shape, dtype=np.float32)
                    
                    chunk_proba_land[valid_mask] = proba[:, 0]
                    chunk_proba_lake[valid_mask] = proba[:, 1]
                    chunk_proba_river[valid_mask] = proba[:, 2]
                    
                    prob_land[y:y+chunk_height, x:x+chunk_width] = chunk_proba_land
                    prob_lake[y:y+chunk_height, x:x+chunk_width] = chunk_proba_lake
                    prob_river[y:y+chunk_height, x:x+chunk_width] = chunk_proba_river
                
                # Make class predictions
                preds = model.predict(X_chunk_scaled)
                
                # Fill in predictions
                chunk_class = np.zeros(valid_mask.shape, dtype=np.uint8)
                chunk_class[valid_mask] = preds
                classification[y:y+chunk_height, x:x+chunk_width] = chunk_class
        
        # Save main classification result
        output_profile = profile.copy()
        output_profile.update(dtype=rasterio.uint8, count=1, nodata=255)
        
        with rasterio.open(output_path, 'w', **output_profile) as dst:
            dst.write(classification, 1)
        
        # Save probability maps if requested
        if probabilities:
            prob_profile = profile.copy()
            prob_profile.update(dtype=rasterio.float32, count=1, nodata=0)
            
            # Save each probability map
            for class_name, class_id in CLASS_MAPPING.items():
                prob_output = output_path.replace(".tif", f"_{class_name}_prob.tif")
                with rasterio.open(prob_output, 'w', **prob_profile) as dst:
                    if class_id == 0:  # land
                        dst.write(prob_land, 1)
                    elif class_id == 1:  # lake
                        dst.write(prob_lake, 1)
                    elif class_id == 2:  # river
                        dst.write(prob_river, 1)
        
        # Create a colored visualization
        viz_output = output_path.replace(".tif", "_viz.png")
        create_visualization(classification, viz_output)
        
        print(f"Classification saved to {output_path}")
        print(f"Visualization saved to {viz_output}")
        
        return classification
            
    except Exception as e:
        print(f"Error applying model to image: {e}")
        raise

In [10]:

def extract_water_boundaries(classification, output_vector_path, class_id=1, min_area_factor=0.01):
    """
    Extract polygon boundaries for water bodies from the classification
    
    Parameters:
    classification: Classification array where water bodies are value class_id
    output_vector_path: Path to save the vector file with water boundaries
    class_id: Class ID to extract (1 for lakes, 2 for rivers)
    min_area_factor: Factor of median area to filter out small polygons
    """
    # Get class name
    class_name = next((name for name, id in CLASS_MAPPING.items() if id == class_id), "unknown")
    
    # Create a binary mask for the specified class
    class_mask = (classification == class_id).astype(np.uint8)
    
    # Get shapes of connected components
    results = (
        {'properties': {'class': class_name}, 'geometry': s}
        for i, (s, v) in enumerate(shapes(class_mask, mask=class_mask == 1))
    )
    
    # Create a GeoDataFrame from the shapes
    gdf = gpd.GeoDataFrame.from_features(list(results), crs="EPSG:4326")
    
    # Filter out very small polygons (likely noise)
    if not gdf.empty:
        gdf['area'] = gdf.geometry.area
        median_area = gdf['area'].median()
        min_area = median_area * min_area_factor
        
        # Filter and report
        original_count = len(gdf)
        gdf = gdf[gdf['area'] > min_area]
        filtered_count = len(gdf)
        
        print(f"Filtered out {original_count - filtered_count} small polygons")
        
        # Save to file
        if not gdf.empty:
            gdf.to_file(output_vector_path)
            print(f"Extracted {len(gdf)} {class_name} polygons saved to {output_vector_path}")
        else:
            print(f"No {class_name} polygons found after filtering")
    else:
        print(f"No {class_name} polygons found in the classification")

In [11]:
def main(band_paths_list, shapefile_path, output_dir, class_field='land_type', model_type='both'):
    """
    Main function to run the entire workflow
    
    Parameters:
    band_paths_list: List of dictionaries, each with paths to blue, green, red, nir, and optionally swir bands
    shapefile_path: Path to the shapefile with polygon classifications
    output_dir: Directory to save outputs
    class_field: Field in the shapefile that contains the class labels
    model_type: 'rf' for Random Forest, 'svm' for SVM, or 'both' for both
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Load polygons
    print("Loading polygon data...")
    polygons_gdf = load_polygons(shapefile_path, class_field)
    print(f"Loaded {len(polygons_gdf)} polygons")
    
    # Create training dataset from all images
    print("Creating training dataset from multiple images...")
    X, y, feature_names = create_training_dataset(band_paths_list, polygons_gdf)
    
    # Train model
    print(f"Training {model_type} model(s)...")
    model_results = train_water_model(X, y, feature_names, model_type=model_type)
    
    # Save the model(s)
    model_filename = os.path.join(output_dir, "water_classification_models.joblib")
    joblib.dump(model_results, model_filename)
    print(f"Model(s) saved to {model_filename}")
    
    # Apply model to each image
    for i, band_paths in enumerate(band_paths_list):
        print(f"\nApplying model to image {i+1}/{len(band_paths_list)}")
        
        # Generate output paths
        image_name = f"image_{i+1}"
        if 'blue' in band_paths:
            # Extract a recognizable name from the blue band path
            image_name = os.path.basename(band_paths['blue']).split('_')[0]
        
        output_classification = os.path.join(output_dir, f"{image_name}_classification.tif")
        
        # Apply the best model (RF if both were trained)
        best_model_type = 'rf' if model_type == 'both' else model_type
        classification = apply_model_to_image(
            model_results, 
            band_paths, 
            output_classification, 
            model_type=best_model_type,
            probabilities=True
        )
        
        # Extract water boundaries for lakes and rivers
        print("Extracting water boundaries...")
        
        # Extract lake boundaries
        lake_output = os.path.join(output_dir, f"{image_name}_lake_boundaries.shp")
        extract_water_boundaries(classification, lake_output, class_id=1)
        
        # Extract river boundaries
        river_output = os.path.join(output_dir, f"{image_name}_river_boundaries.shp")
        extract_water_boundaries(classification, river_output, class_id=2)
    
    print("\nProcessing complete!")

# Example usage:
if __name__ == "__main__":
    # Define paths to Sentinel-2 bands for each image
    # These are direct paths to each band JP2 file
    band_paths_list = [
        {
            # Image 1
            'blue': "/Users/chloe/Documents/INDSTUDY/Training Data Images/Polygon_Extraction_images/11.SAFE/GRANULE/L2A_T05VNJ_A033331_20230724T214533/IMG_DATA/R10m/T05VNJ_20230724T214539_B02_10m.jp2",  # Blue band
            'green': "/Users/chloe/Documents/INDSTUDY/Training Data Images/Polygon_Extraction_images/11.SAFE/GRANULE/L2A_T05VNJ_A033331_20230724T214533/IMG_DATA/R10m/T05VNJ_20230724T214539_B03_10m.jp2", # Green band
            'red': "/Users/chloe/Documents/INDSTUDY/Training Data Images/Polygon_Extraction_images/11.SAFE/GRANULE/L2A_T05VNJ_A033331_20230724T214533/IMG_DATA/R10m/T05VNJ_20230724T214539_B04_10m.jp2",   # Red band
            'nir': "/Users/chloe/Documents/INDSTUDY/Training Data Images/Polygon_Extraction_images/11.SAFE/GRANULE/L2A_T05VNJ_A033331_20230724T214533/IMG_DATA/R10m/T05VNJ_20230724T214539_B08_10m.jp2"   # NIR band
        },
        {
            # Image 2
            'blue': "/Users/chloe/Documents/INDSTUDY/Training Data Images/Polygon_Extraction_images/12.SAFE/GRANULE/L2A_T05VNK_A033288_20230721T213533/IMG_DATA/R10m/T05VNK_20230721T213539_B02_10m.jp2",
            'green': "/Users/chloe/Documents/INDSTUDY/Training Data Images/Polygon_Extraction_images/12.SAFE/GRANULE/L2A_T05VNK_A033288_20230721T213533/IMG_DATA/R10m/T05VNK_20230721T213539_B03_10m.jp2",
            'red': "/Users/chloe/Documents/INDSTUDY/Training Data Images/Polygon_Extraction_images/12.SAFE/GRANULE/L2A_T05VNK_A033288_20230721T213533/IMG_DATA/R10m/T05VNK_20230721T213539_B04_10m.jp2",
            'nir': "/Users/chloe/Documents/INDSTUDY/Training Data Images/Polygon_Extraction_images/12.SAFE/GRANULE/L2A_T05VNK_A033288_20230721T213533/IMG_DATA/R10m/T05VNK_20230721T213539_B08_10m.jp2"
        },
        {
            # Image 3
            'blue': "/Users/chloe/Documents/INDSTUDY/Training Data Images/Polygon_Extraction_images/21.SAFE/GRANULE/L2A_T05VPJ_A042225_20230723T212524/IMG_DATA/R10m/T05VPJ_20230723T212521_B02_10m.jp2",
            'green': "/Users/chloe/Documents/INDSTUDY/Training Data Images/Polygon_Extraction_images/21.SAFE/GRANULE/L2A_T05VPJ_A042225_20230723T212524/IMG_DATA/R10m/T05VPJ_20230723T212521_B03_10m.jp2",
            'red': "/Users/chloe/Documents/INDSTUDY/Training Data Images/Polygon_Extraction_images/21.SAFE/GRANULE/L2A_T05VPJ_A042225_20230723T212524/IMG_DATA/R10m/T05VPJ_20230723T212521_B04_10m.jp2",
            'nir': "/Users/chloe/Documents/INDSTUDY/Training Data Images/Polygon_Extraction_images/21.SAFE/GRANULE/L2A_T05VPJ_A042225_20230723T212524/IMG_DATA/R10m/T05VPJ_20230723T212521_B08_10m.jp2"
        },
        {
            # Image 4
            'blue': "/Users/chloe/Documents/INDSTUDY/Training Data Images/Polygon_Extraction_images/22.SAFE/GRANULE/L2A_T06VUQ_A042411_20230805T213615/IMG_DATA/R10m/T06VUQ_20230805T213531_B02_10m.jp2",
            'green': "/Users/chloe/Documents/INDSTUDY/Training Data Images/Polygon_Extraction_images/22.SAFE/GRANULE/L2A_T06VUQ_A042411_20230805T213615/IMG_DATA/R10m/T06VUQ_20230805T213531_B03_10m.jp2",
            'red': "/Users/chloe/Documents/INDSTUDY/Training Data Images/Polygon_Extraction_images/22.SAFE/GRANULE/L2A_T06VUQ_A042411_20230805T213615/IMG_DATA/R10m/T06VUQ_20230805T213531_B04_10m.jp2",
            'nir': "/Users/chloe/Documents/INDSTUDY/Training Data Images/Polygon_Extraction_images/22.SAFE/GRANULE/L2A_T06VUQ_A042411_20230805T213615/IMG_DATA/R10m/T06VUQ_20230805T213531_B08_10m.jp2"
        },
        {
            # Image 5
            'blue': "/Users/chloe/Documents/INDSTUDY/Training Data Images/Polygon_Extraction_images/31.SAFE/GRANULE/L2A_T06VWP_A034246_20230926T212653/IMG_DATA/R10m/T06VWP_20230926T212529_B02_10m.jp2",
            'green': "/Users/chloe/Documents/INDSTUDY/Training Data Images/Polygon_Extraction_images/31.SAFE/GRANULE/L2A_T06VWP_A034246_20230926T212653/IMG_DATA/R10m/T06VWP_20230926T212529_B03_10m.jp2",
            'red': "/Users/chloe/Documents/INDSTUDY/Training Data Images/Polygon_Extraction_images/31.SAFE/GRANULE/L2A_T06VWP_A034246_20230926T212653/IMG_DATA/R10m/T06VWP_20230926T212529_B04_10m.jp2",
            'nir': "/Users/chloe/Documents/INDSTUDY/Training Data Images/Polygon_Extraction_images/31.SAFE/GRANULE/L2A_T06VWP_A034246_20230926T212653/IMG_DATA/R10m/T06VWP_20230926T212529_B08_10m.jp2"
        }
        # {
        #     # Image 6
        #     'blue': "/Users/chloe/Documents/INDSTUDY/Training Data Images/Polygon_Extraction_images/32.SAFE/GRANULE/L2A_T06VWQ_A034246_20230926T212653/IMG_DATA/R10m/T06VWQ_20230926T212529_B02_10m.jp2",
        #     'green': "/Users/chloe/Documents/INDSTUDY/Training Data Images/Polygon_Extraction_images/21.SAFE/GRANULE/L2A_T05VPJ_A042225_20230723T212524/IMG_DATA/R10m/T05VPJ_20230723T212521_B03_10m.jp2",
        #     'red': "/Users/chloe/Documents/INDSTUDY/Training Data Images/Polygon_Extraction_images/21.SAFE/GRANULE/L2A_T05VPJ_A042225_20230723T212524/IMG_DATA/R10m/T05VPJ_20230723T212521_B04_10m.jp2",
        #     'nir': "/Users/chloe/Documents/INDSTUDY/Training Data Images/Polygon_Extraction_images/21.SAFE/GRANULE/L2A_T05VPJ_A042225_20230723T212524/IMG_DATA/R10m/T05VPJ_20230723T212521_B08_10m.jp2"
        # }
    ]
    
    # Define path to your shapefile with polygon classifications
    polygons_path = "/Users/chloe/Documents/INDSTUDY/Training Data Images/Land_Type_Training/Training_b.shp"
    
    # Define output directory
    output_dir = "/Users/chloe/Documents/INDSTUDY/Training Data Images/Land_type_trainind/Output test classifications"
    
    # Run the workflow
    main(
        band_paths_list=band_paths_list,
        shapefile_path=polygons_path,
        output_dir=output_dir,
        class_field='land_type',  # Change to match your shapefile's attribute field
        model_type='both'         # Train both RF and SVM models
    )

Loading polygon data...
Shapefile columns: ['id', 'land_type', 'geometry']
Unique values in land_type: ['lake' 'land' 'river']
Loaded 101 polygons
Creating training dataset from multiple images...
Processing image set 1/5
Successfully loaded blue band from T05VNJ_20230724T214539_B02_10m.jp2
Successfully loaded green band from T05VNJ_20230724T214539_B03_10m.jp2
Successfully loaded red band from T05VNJ_20230724T214539_B04_10m.jp2
Successfully loaded nir band from T05VNJ_20230724T214539_B08_10m.jp2
Found 58 overlapping polygons


Extracting from image set 1: 100%|██████████| 58/58 [00:00<00:00, 10962.04it/s]

Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' o

Successfully loaded blue band from T05VNK_20230721T213539_B02_10m.jp2
Successfully loaded green band from T05VNK_20230721T213539_B03_10m.jp2
Successfully loaded red band from T05VNK_20230721T213539_B04_10m.jp2
Successfully loaded nir band from T05VNK_20230721T213539_B08_10m.jp2
Found 101 overlapping polygons


Extracting from image set 2: 100%|██████████| 101/101 [00:00<00:00, 16895.65it/s]

...

Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' o

Successfully loaded blue band from T06VWP_20230926T212529_B02_10m.jp2
Successfully loaded green band from T06VWP_20230926T212529_B03_10m.jp2
Successfully loaded red band from T06VWP_20230926T212529_B04_10m.jp2
Successfully loaded nir band from T06VWP_20230926T212529_B08_10m.jp2
Found 2 overlapping polygons


Extracting from image set 5: 100%|██████████| 2/2 [00:00<00:00, 5953.59it/s]

Error processing polygon: 'dict' object has no attribute 'transform'
Error processing polygon: 'dict' object has no attribute 'transform'


ValueError: No valid features could be extracted from any of the images